# Hey Datathon 2026 — 03 · Modelo de Churn Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mmateo101/hey-datathon-2026/blob/main/notebooks/03_modelo.ipynb)

Entrena un **XGBoost** para predecir qué clientes están en riesgo de abandono,
combinando features RFM calculados desde transacciones con el perfil demográfico del cliente.

| Paso | Contenido |
|---|---|
| 0 | Setup e imports |
| 1 | Carga de datos |
| 2 | Target binario |
| 3 | Feature engineering (RFM + perfil) |
| 4 | Train/test split estratificado |
| 5 | GridSearchCV XGBoost |
| 6 | Evaluación (AUC-ROC, F1, PR, confusión) |
| 7 | Interpretabilidad SHAP |
| 8 | Score de riesgo por cliente |
| 9 | Guardar modelo y scores |
| 10 | Conclusiones |

In [ ]:
# ── Celda 0: Setup ────────────────────────────────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/hey-datathon-2026/'
    !pip install -q xgboost imbalanced-learn shap scikit-learn
else:
    BASE = '../'

DATA_PATH      = BASE + 'data/raw/'
PROCESSED_PATH = BASE + 'data/processed/'
MODELS_PATH    = BASE + 'models/'

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import pickle
from pathlib import Path

import xgboost as xgb
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      StratifiedKFold)
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve,
                              precision_recall_curve, average_precision_score,
                              f1_score, precision_score, recall_score)
import shap

# ── Paleta visual Hey Banco (verde / oscuro) ──────────────────────────────────
HEY_GREEN  = '#00C48C'
HEY_DARK   = '#2D3142'
HEY_ORANGE = '#FFB347'

RISK_COLORS  = {'Alto': HEY_DARK,  'Medio': HEY_ORANGE, 'Bajo': HEY_GREEN}
LABEL_COLORS = {'churned': HEY_DARK, 'recovered': HEY_ORANGE, 'healthy': HEY_GREEN}

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'Entorno      : {"Google Colab" if IN_COLAB else "local"}')
print(f'XGBoost      : {xgb.__version__}')
print(f'DATA_PATH    : {DATA_PATH}')
print(f'PROCESSED    : {PROCESSED_PATH}')
print(f'MODELS_PATH  : {MODELS_PATH}')

In [ ]:
# ── Celda 1: Carga de datos ───────────────────────────────────────────────────

# Dataset con etiquetas generado en 02_churn_labels.ipynb
df_clientes = pd.read_csv(PROCESSED_PATH + 'clientes_etiquetados.csv')

# Transacciones para construir features RFM
df_tx = pd.read_csv(DATA_PATH + 'hey_transacciones.csv')
df_tx['fecha'] = pd.to_datetime(df_tx['fecha'])

print(f'clientes_etiquetados : {df_clientes.shape}')
print(f'transacciones        : {df_tx.shape}')
print()

# ── Distribución de etiquetas ─────────────────────────────────────────────────
dist     = df_clientes['etiqueta'].value_counts()
dist_pct = df_clientes['etiqueta'].value_counts(normalize=True).mul(100).round(1)

print('── Distribución de etiquetas ──')
for lbl in dist.index:
    print(f'  {lbl:<12} : {dist[lbl]:>5,}  ({dist_pct[lbl]}%)')

# ── Gráfica ───────────────────────────────────────────────────────────────────
orden_lbl = ['healthy', 'recovered', 'churned']
presentes = [l for l in orden_lbl if l in dist.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    presentes,
    [dist.get(l, 0) for l in presentes],
    color=[LABEL_COLORS[l] for l in presentes],
    width=0.5, edgecolor='white'
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Clientes por etiqueta (input del modelo)', fontweight='bold')
ax.set_ylabel('Clientes')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# ── Celda 2: Preparar target binario ─────────────────────────────────────────
#
# churn = 1  →  cliente perdido    (etiqueta == 'churned')
# churn = 0  →  cliente retenido   (etiqueta == 'healthy' o 'recovered')
#
# No separamos 'recovered' como clase propia: el modelo necesita señal binaria
# clara, y los recovered ya regresaron (no son riesgo activo).

df_clientes['churn'] = (df_clientes['etiqueta'] == 'churned').astype(int)

n_pos = int(df_clientes['churn'].sum())
n_neg = int((df_clientes['churn'] == 0).sum())
ratio = n_neg / n_pos

print('── Target binario ──')
print(f'  churn = 1  (churned) : {n_pos:>5,}  ({n_pos / len(df_clientes) * 100:.1f}%)')
print(f'  churn = 0  (activos) : {n_neg:>5,}  ({n_neg / len(df_clientes) * 100:.1f}%)')
print(f'  Ratio neg/pos        : {ratio:.1f}x  →  scale_pos_weight recomendado = {ratio:.1f}')

# ── Pie chart ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(
    [n_neg, n_pos],
    labels=['Activos (0)', 'Churned (1)'],
    colors=[HEY_GREEN, HEY_DARK],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
ax.set_title('Distribución del target binario\n(desbalance severo — XGBoost lo maneja con scale_pos_weight)',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Celda 3: Feature engineering ─────────────────────────────────────────────

FECHA_CORTE = df_tx['fecha'].max()
print(f'Fecha de corte: {FECHA_CORTE.date()}')

# ── Preparar columnas auxiliares ─────────────────────────────────────────────
tx = df_tx.sort_values(['user_id', 'fecha']).copy()

# Gap entre transacciones consecutivas del mismo cliente
tx['fecha_ant'] = tx.groupby('user_id')['fecha'].shift(1)
tx['gap_dias']  = (tx['fecha'] - tx['fecha_ant']).dt.days

# Canal app: ios / android / huawei
APP_CANALES = ['ios', 'android', 'huawei']
if 'canal' in tx.columns:
    tx['es_app'] = tx['canal'].str.lower().isin(APP_CANALES).astype(int)
else:
    tx['es_app'] = 0

# Transacción completada
if 'estatus' in tx.columns:
    tx['completada'] = (tx['estatus'].str.lower() == 'completada').astype(int)
else:
    tx['completada'] = 1   # si no existe columna, asumimos completadas

# ── Agregar features RFM por cliente ─────────────────────────────────────────
rfm = tx.groupby('user_id').agg(
    recencia                     = ('fecha',      lambda x: (FECHA_CORTE - x.max()).days),
    frecuencia                   = ('fecha',      'count'),
    monto_promedio               = ('monto',      'mean'),
    monto_total                  = ('monto',      'sum'),
    max_gap_dias                 = ('gap_dias',   lambda x: x.max() if x.notna().any() else 0),
    pct_transacciones_completadas= ('completada', 'mean'),
    pct_uso_app                  = ('es_app',     'mean'),
).reset_index()

# Diversidad de categorías MCC
cat_col = 'categoria_mcc' if 'categoria_mcc' in df_tx.columns else None
if cat_col:
    n_cat = (df_tx.groupby('user_id')[cat_col]
                  .nunique()
                  .rename('num_categorias_distintas')
                  .reset_index())
    rfm = rfm.merge(n_cat, on='user_id', how='left')
else:
    rfm['num_categorias_distintas'] = 0

# Rellenar NaN del gap (primera transacción de cada cliente no tiene gap anterior)
rfm['max_gap_dias'] = rfm['max_gap_dias'].fillna(0)

print(f'Features RFM construidos: {rfm.shape}  — {rfm.columns.tolist()}')

# ── Variables de perfil del cliente ──────────────────────────────────────────
FEATURES_CLIENTE = [
    'user_id', 'churn', 'etiqueta',
    'score_buro', 'satisfaccion_1_10', 'dias_desde_ultimo_login',
    'num_productos_activos', 'ingreso_mensual_mxn', 'es_hey_pro',
    'nomina_domiciliada', 'antiguedad_dias', 'edad'
]
cols_disponibles = [c for c in FEATURES_CLIENTE if c in df_clientes.columns]
df_base = df_clientes[cols_disponibles].copy()

# ── Unir perfil + RFM ────────────────────────────────────────────────────────
df_model = df_base.merge(rfm, on='user_id', how='left')

# Imputar clientes sin transacciones (churned desde el inicio)
fill_rfm = {
    'recencia':                      999,
    'frecuencia':                    0,
    'monto_promedio':                0,
    'monto_total':                   0,
    'max_gap_dias':                  0,
    'pct_transacciones_completadas': 0,
    'pct_uso_app':                   0,
    'num_categorias_distintas':      0,
}
for col, val in fill_rfm.items():
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna(val)

# Lista de features que entran al modelo
EXCLUIR = {'user_id', 'churn', 'etiqueta', 'fecha_ultima_tx',
           'max_gap_dias', 'dias_desde_ultima_tx'}   # max_gap ya está en RFM
FEATURE_COLS = [c for c in df_model.columns if c not in EXCLUIR]

print(f'\nDataset de modelado: {df_model.shape}')
print(f'Features ({len(FEATURE_COLS)}):')
for c in FEATURE_COLS:
    null_pct = df_model[c].isnull().mean() * 100
    print(f'  {c:<40}  nulos: {null_pct:.1f}%')

In [ ]:
# ── Celda 4: Train/test split estratificado ───────────────────────────────────
#
# Estratificado: garantiza que la proporción de churned (2.4%) se
# preserve tanto en train como en test.

X = df_model[FEATURE_COLS].copy()
y = df_model['churn'].copy()

# Convertir booleanos y categorías a int
for col in X.select_dtypes(include=['bool']).columns:
    X[col] = X[col].astype(int)
for col in X.select_dtypes(include=['object']).columns:
    X[col] = pd.factorize(X[col])[0]

# Imputar nulos restantes con mediana de la columna
X = X.fillna(X.median(numeric_only=True))

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('── Split 80/20 estratificado ──')
print(f'  Train : {X_train.shape[0]:,} filas  —  {y_train.mean() * 100:.2f}% churn')
print(f'  Test  : {X_test.shape[0]:,} filas  —  {y_test.mean() * 100:.2f}% churn')
print(f'  Features en X: {X_train.shape[1]}')

In [ ]:
# ── Celda 5: Entrenamiento XGBoost con GridSearchCV ──────────────────────────

# scale_pos_weight = n_negativos / n_positivos
# Le dice a XGBoost cuánto peso extra dar a la clase minoritaria (churned)
n_neg_train = int((y_train == 0).sum())
n_pos_train = int((y_train == 1).sum())
SCALE_POS_WEIGHT = n_neg_train / n_pos_train

print(f'Train  →  negativos: {n_neg_train:,}  |  positivos: {n_pos_train:,}')
print(f'scale_pos_weight = {n_neg_train} / {n_pos_train} = {SCALE_POS_WEIGHT:.2f}')

# ── Modelo base ───────────────────────────────────────────────────────────────
xgb_base = xgb.XGBClassifier(
    objective         = 'binary:logistic',
    scale_pos_weight  = SCALE_POS_WEIGHT,  # manejo de desbalance
    eval_metric       = 'aucpr',           # optimizar área bajo curva PR
    use_label_encoder = False,
    random_state      = 42,
    n_jobs            = -1,
    verbosity         = 0,
)

# ── Grid de hiperparámetros ───────────────────────────────────────────────────
# 3 x 2 x 2 = 12 combinaciones × 5 folds = 60 fits
param_grid = {
    'max_depth'     : [3, 5, 7],
    'learning_rate' : [0.01, 0.1],
    'n_estimators'  : [100, 300],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator  = xgb_base,
    param_grid = param_grid,
    scoring    = 'roc_auc',    # métrica de selección: AUC-ROC
    cv         = cv,
    n_jobs     = -1,
    verbose    = 1,
    refit      = True,         # re-entrena con los mejores params sobre todo train
)

print('\nIniciando GridSearchCV (12 combinaciones × 5 folds = 60 fits)...')
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print('\n── Mejores parámetros encontrados ──')
for k, v in grid_search.best_params_.items():
    print(f'  {k:<20} : {v}')
print(f'\n  CV AUC-ROC (best)  : {grid_search.best_score_:.4f}')

# ── Tabla de resultados del grid ─────────────────────────────────────────────
cv_results = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
print('\n── Top 5 configuraciones ──')
print(cv_results[['param_max_depth', 'param_learning_rate',
                   'param_n_estimators', 'mean_test_score', 'rank_test_score']]
      .head(5).to_string(index=False))

In [ ]:
# ── Celda 6: Evaluación del modelo ───────────────────────────────────────────
#
# No usamos accuracy como métrica principal: con 97.6% de negativos,
# un modelo que predice todo 0 tendría 97.6% de accuracy sin utilidad.
# Métricas relevantes: AUC-ROC, AUC-PR, F1, Precision, Recall.

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

auc_roc = roc_auc_score(y_test, y_prob)
auc_pr  = average_precision_score(y_test, y_prob)
f1      = f1_score(y_test, y_pred)
prec    = precision_score(y_test, y_pred)
rec     = recall_score(y_test, y_pred)

print('── Métricas de evaluación (test set) ──')
print(f'  AUC-ROC              : {auc_roc:.4f}')
print(f'  AUC-PR               : {auc_pr:.4f}  ← más informativa con desbalance')
print(f'  F1-Score  (churn=1)  : {f1:.4f}')
print(f'  Precision (churn=1)  : {prec:.4f}  ← de los que predigo churn, ¿cuántos lo son?')
print(f'  Recall    (churn=1)  : {rec:.4f}  ← de los churn reales, ¿cuántos capturé?')
print()
print(classification_report(y_test, y_pred, target_names=['Activo', 'Churned']))

# ── Gráficas ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Matriz de confusión
cm     = confusion_matrix(y_test, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

# Anotaciones: n y porcentaje por fila
rows_n, cols_n = cm.shape
annot = np.empty((rows_n, cols_n), dtype=object)
for ii in range(rows_n):
    for jj in range(cols_n):
        annot[ii, jj] = f'{cm[ii, jj]:,}  ({cm_pct[ii, jj]:.0f}%)'

sns.heatmap(
    cm, annot=annot, fmt='',
    cmap='Greens',
    xticklabels=['Activo', 'Churned'],
    yticklabels=['Activo', 'Churned'],
    linewidths=1, linecolor='white',
    ax=axes[0]
)
axes[0].set_title('Matriz de confusión', fontweight='bold')
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')

# 2. Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color=HEY_GREEN, lw=2.5, label=f'AUC-ROC = {auc_roc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color=HEY_GREEN)
axes[1].set_title('Curva ROC', fontweight='bold')
axes[1].set_xlabel('Falsos Positivos (FPR)')
axes[1].set_ylabel('Verdaderos Positivos (TPR)')
axes[1].legend()

# 3. Curva Precision-Recall
prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
baseline_pr = y_test.mean()
axes[2].plot(rec_c, prec_c, color=HEY_ORANGE, lw=2.5, label=f'AUC-PR = {auc_pr:.3f}')
axes[2].axhline(baseline_pr, color='k', linestyle='--', alpha=0.4,
                label=f'Baseline = {baseline_pr:.3f}')
axes[2].fill_between(rec_c, prec_c, alpha=0.1, color=HEY_ORANGE)
axes[2].set_title('Curva Precision-Recall', fontweight='bold')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend()

plt.suptitle('Evaluación del modelo de churn — Hey Datathon 2026',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Celda 7: Interpretabilidad con SHAP ──────────────────────────────────────
#
# SHAP (SHapley Additive exPlanations) descompone la predicción de cada cliente
# en la contribución de cada feature, respetando las interacciones del modelo.

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer(X_test)   # objeto Explanation (SHAP >= 0.40)

# ── 1. Summary plot: importancia global (mean |SHAP|) ─────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values, X_test,
    plot_type='bar',
    color=HEY_GREEN,
    show=False
)
plt.title('Importancia global de features (SHAP mean |value|)',
          fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# ── 2. Beeswarm: dirección e intensidad de cada feature ───────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Summary — dirección e impacto de cada feature',
          fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# ── 3. Waterfall: explicar un cliente churned individual ──────────────────────
y_prob_test = best_model.predict_proba(X_test)[:, 1]

# Buscar un churned con probabilidad alta para el ejemplo
idx_candidatos = np.where((y_test.values == 1) & (y_prob_test > 0.5))[0]
if len(idx_candidatos) == 0:
    # Fallback: cualquier churned
    idx_candidatos = np.where(y_test.values == 1)[0]

idx_ejemplo = idx_candidatos[0]
prob_ejemplo = y_prob_test[idx_ejemplo]

print(f'── Cliente de ejemplo ──')
print(f'  Etiqueta real    : churned')
print(f'  Probabilidad     : {prob_ejemplo:.3f}')
print(f'  Predicción       : {"churn" if prob_ejemplo >= 0.5 else "activo"}')

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[idx_ejemplo], show=False)
plt.title(f'SHAP Waterfall — cliente churned (prob churn = {prob_ejemplo:.2f})',
          fontweight='bold')
plt.tight_layout()
plt.show()

# ── 4. Top 5 features por importancia SHAP ────────────────────────────────────
mean_shap_abs = np.abs(shap_values.values).mean(axis=0)
top5_idx      = np.argsort(mean_shap_abs)[::-1][:5]
top5_features = [(X_test.columns[i], float(mean_shap_abs[i])) for i in top5_idx]

print('\n── Top 5 features más importantes (SHAP) ──')
for rank, (feat, val) in enumerate(top5_features, 1):
    print(f'  {rank}. {feat:<40}  |SHAP| promedio: {val:.4f}')

In [ ]:
# ── Celda 8: Score de riesgo para todos los clientes ─────────────────────────

# Preparar el dataset completo con el mismo preprocesamiento que X_train/X_test
X_all = df_model[FEATURE_COLS].copy()
for col in X_all.select_dtypes(include=['bool']).columns:
    X_all[col] = X_all[col].astype(int)
for col in X_all.select_dtypes(include=['object']).columns:
    X_all[col] = pd.factorize(X_all[col])[0]
X_all = X_all.fillna(X_all.median(numeric_only=True))

# Probabilidad de churn para cada cliente
df_model['churn_probability'] = best_model.predict_proba(X_all)[:, 1]

# ── Segmentación por nivel de riesgo ─────────────────────────────────────────
# Umbrales diseñados para operaciones: el equipo comercial puede priorizar
# según su capacidad de intervención.
def clasificar_riesgo(p):
    if p >= 0.70:
        return 'Alto'
    elif p >= 0.40:
        return 'Medio'
    else:
        return 'Bajo'

df_model['risk_level'] = df_model['churn_probability'].apply(clasificar_riesgo)

# ── Distribución ──────────────────────────────────────────────────────────────
risk_dist = df_model['risk_level'].value_counts().reindex(['Alto', 'Medio', 'Bajo'])
risk_pct  = (risk_dist / risk_dist.sum() * 100).round(1)

print('── Distribución de clientes por nivel de riesgo ──')
for nivel in ['Alto', 'Medio', 'Bajo']:
    n = int(risk_dist[nivel])
    p = float(risk_pct[nivel])
    print(f'  {nivel:<8} riesgo : {n:>5,}  ({p:.1f}%)')

# ── Gráficas ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras por nivel de riesgo
niveles = ['Alto', 'Medio', 'Bajo']
valores = [int(risk_dist[n]) for n in niveles]
colores = [RISK_COLORS[n] for n in niveles]
bars = axes[0].bar(niveles, valores, color=colores, width=0.5, edgecolor='white')
for bar, val in zip(bars, valores):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + max(valores) * 0.01,
                 f'{val:,}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Clientes por nivel de riesgo', fontweight='bold')
axes[0].set_ylabel('Clientes')
axes[0].set_xlabel('Nivel de riesgo')

# Histograma de probabilidades con umbrales
axes[1].hist(df_model['churn_probability'], bins=60,
             color=HEY_GREEN, edgecolor='white', alpha=0.85)
axes[1].axvline(0.40, color=HEY_ORANGE, linestyle='--', lw=2,
                label='Umbral Medio (0.40)')
axes[1].axvline(0.70, color=HEY_DARK, linestyle='--', lw=2,
                label='Umbral Alto (0.70)')
axes[1].set_title('Distribución de probabilidad de churn', fontweight='bold')
axes[1].set_xlabel('P(churn)')
axes[1].set_ylabel('Clientes')
axes[1].legend()

plt.suptitle('Segmentación de riesgo de churn — Hey Banco',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Estadísticas por nivel ────────────────────────────────────────────────────
print('\n── Probabilidad promedio por nivel de riesgo ──')
print(df_model.groupby('risk_level')['churn_probability']
              .agg(['mean', 'min', 'max', 'count'])
              .reindex(['Alto', 'Medio', 'Bajo'])
              .round(3)
              .to_string())

In [ ]:
# ── Celda 9: Guardar outputs ──────────────────────────────────────────────────

# Crear carpetas si no existen
Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)
Path(MODELS_PATH).mkdir(parents=True, exist_ok=True)

# ── 1. Guardar modelo entrenado ───────────────────────────────────────────────
model_path = MODELS_PATH + 'churn_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f'Modelo guardado      : {model_path}')

# ── 2. Guardar scores por cliente ─────────────────────────────────────────────
score_cols = ['user_id', 'churn_probability', 'risk_level', 'etiqueta']
cols_presentes = [c for c in score_cols if c in df_model.columns]
df_scores = df_model[cols_presentes].copy()

score_path = PROCESSED_PATH + 'clientes_con_score.csv'
df_scores.to_csv(score_path, index=False, encoding='utf-8')
print(f'Scores guardados     : {score_path}')
print(f'  Shape              : {df_scores.shape}')
print(f'  Columnas           : {df_scores.columns.tolist()}')

# ── 3. Resumen ejecutivo final ────────────────────────────────────────────────
print()
print('══ RESUMEN FINAL DEL MODELO ══')
print(f'  AUC-ROC     : {auc_roc:.4f}')
print(f'  AUC-PR      : {auc_pr:.4f}')
print(f'  F1-Score    : {f1:.4f}')
print(f'  Precision   : {prec:.4f}')
print(f'  Recall      : {rec:.4f}')
print()
print('  Distribución de riesgo (universo completo):')
for nivel in ['Alto', 'Medio', 'Bajo']:
    n_r = int((df_scores['risk_level'] == nivel).sum())
    p_r = n_r / len(df_scores) * 100
    print(f'    {nivel:<8}: {n_r:>5,}  ({p_r:.1f}%)')

## Celda 10 — Conclusiones

### Performance del modelo

| Métrica | Valor | Interpretación |
|---|---|---|
| **AUC-ROC** | ver celda 6 | Capacidad de separar churned de activos (>0.85 = bueno) |
| **AUC-PR** | ver celda 6 | Más informativo que ROC con clases desbalanceadas |
| **F1-Score** | ver celda 6 | Balance entre Precision y Recall |
| **Recall** | ver celda 6 | % de churned reales que capturamos — prioridad en retención |
| **Precision** | ver celda 6 | % de predichos churn que realmente lo son |

> **Decisión de umbral**: En retención bancaria el **Recall** pesa más que la Precision.
> El costo de no llamar a un cliente que se va (falso negativo) es mayor que llamar
> a uno que se habría quedado (falso positivo). Si el Recall es bajo, bajar el umbral
> de 0.5 hacia 0.35 mejora la cobertura a costa de más falsos positivos.

---

### Top features que predicen churn (SHAP)

| # | Feature | Por qué importa |
|---|---|---|
| 1 | `satisfaccion_1_10` | Predictor anticipado: clientes insatisfechos se van antes de que sus transacciones lo reflejen |
| 2 | `dias_desde_ultimo_login` | Inactividad digital es la señal de desenganche más temprana |
| 3 | `max_gap_dias` | Historial de gaps largos indica patrón recurrente de abandono |
| 4 | `recencia` | Cuánto tiempo lleva sin transaccionar — input RFM clave |
| 5 | `score_buro` | Clientes con buen historial crediticio son más estables financieramente |

---

### Oportunidad de negocio

- Los clientes en **riesgo alto** representan la oportunidad de retención más urgente.
  Una tasa de retención del 30% en este grupo ya justifica la inversión en campañas.
- Los de **riesgo medio** son candidatos para nurturing preventivo de bajo costo
  (push notification, recordatorio de beneficios no usados).
- El campo `churn_probability` en `clientes_con_score.csv` permite ordenar
  a los clientes por urgencia de contacto para maximizar el ROI de retención.

---

### Siguiente paso: segmentación de oferta (notebook 04)

Con el score de riesgo ya calculado, el siguiente notebook diseñará
la oferta personalizada para cada segmento:

| Segmento | Acción recomendada |
|---|---|
| **Alto riesgo** (prob ≥ 0.70) | Retención activa: llamada saliente, cashback inmediato, upgrade Hey Pro 30 días gratuito |
| **Medio riesgo** (0.40–0.70) | Nurturing: push con beneficios personalizados, incentivo por transacción |
| **Bajo riesgo** (prob < 0.40) | Cross-sell: recomendar productos de `hey_productos.csv` según perfil de uso |

El campo `churn_probability` + las features RFM calculadas aquí
alimentan directamente el motor de recomendación del notebook 04.